In [27]:
from sklearn.model_selection import train_test_split as split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import fbeta_score, precision_score, recall_score, make_scorer
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import pandas as pd
from sklearn.metrics import accuracy_score
import pickle

In [28]:
df = pd.read_csv("../data/processed/f1_strategy_dataset_v4_processed.csv")

In [29]:
X = df.drop('PitNextLap', axis=1)
y = df['PitNextLap']

X_train, X_test, y_train, y_test = split(X, y, test_size=0.2, random_state=42)
print(f"Доля пит-стопов в трейне: {y_train.mean():.4f}")
print(f"Доля пит-стопов в тесте: {y_test.mean():.4f}")

Доля пит-стопов в трейне: 0.2539
Доля пит-стопов в тесте: 0.2583


In [30]:
scaler = StandardScaler()
scaler.fit(X_train)

X_test_scaled = scaler.transform(X_test)
X_train_scaled = scaler.transform(X_train)

X_train = X_train_scaled
X_test = X_test_scaled

In [31]:
def print_metrics(model_name, y_true, y_pred):    
    f2 = fbeta_score(y_true, y_pred, beta=2)
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    
    print("="*100)
    print(f"Model name: {model_name}")
    print(f"F2-score: {f2:.2f}")
    print(f"Accuracy: {acc:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print("="*100)

In [32]:
f2_scorer = make_scorer(fbeta_score, beta=2)

In [33]:
def train_grid_search(model_name, trainer, X_train, y_train, X_test, y_test):
    gs = trainer(X_train, y_train)
    
    best_model = gs.best_estimator_
    y_pred = best_model.predict(X_test)
    y_pred_train = best_model.predict(X_train)
    
    f2 = fbeta_score(y_test, y_pred, beta=2)
    acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='macro')
    recall = recall_score(y_test, y_pred, average='macro')
    
    f2_train = fbeta_score(y_test, y_pred, beta=2)
    acc_train = accuracy_score(y_test, y_pred)
    precision_train = precision_score(y_test, y_pred, average='macro')
    recall_train = recall_score(y_test, y_pred, average='macro')
    
    print("="*100)
    print(f"Model name: {model_name}")
    print(f'Params: {gs.best_params_}')
    print(f"F2-score: {f2:.2f}")
    print(f"Accuracy: {acc:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print("="*100)
    
    return gs.best_estimator_

In [34]:
def train_logreg(X_train, y_train):
    model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

    param_grid = {
        'C': [0.01, 0.1, 1, 10, 100],
        'solver': ['liblinear', 'saga'],
        'l1_ratio': [0,1]
    }

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=f2_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    return grid_search

In [35]:
def train_svm(X_train, y_train) -> GridSearchCV:
    model = SVC(max_iter=1000, random_state=42, kernel='linear', class_weight='balanced', probability=True)

    param_grid = {
        'C': [0.01, 0.1, 1, 10, 100]
    }
    
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=f2_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    return grid_search

In [36]:
def train_random_forest(X_train, y_train):
    model = RandomForestClassifier(random_state=42, class_weight='balanced_subsample')

    param_grid = {
        'n_estimators': [1, 5, 10, 50, 100],
        'max_depth': [10, 25, 100, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4] 
    }

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=f2_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    return grid_search

In [37]:
def train_xgboost(X_train, y_train):
    model = XGBClassifier(random_state=42, objective='binary:logistic')

    param_grid = {
        'objective': ['binary:logistic'],
        'max_depth': [3, 5, 10],
        'learning_rate': [0.1, 0.01, 0.001],
        'n_estimators': [50, 100, 500, 1000]
    }

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=f2_scorer,
        cv=5,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    grid_search.fit(X_train, y_train)

    return grid_search

In [38]:
logreg = LogisticRegression()
logreg.fit(X_train_scaled, y_train)
y_pred = logreg.predict(X_test_scaled)
print_metrics('Base logreg', y_test, y_pred)
with open('../models/base_logreg.pkl', 'wb') as f:
    pickle.dump(logreg, f)

Model name: Base logreg
F2-score: 0.33
Accuracy: 0.77
Precision: 0.70
Recall: 0.62


In [39]:
# best_logreg = train_grid_search('Logistic regression', train_logreg, X_train_scaled, y_train, X_test, y_test)
# with open('../models/logreg.pkl', 'wb') as f:
#     pickle.dump(best_logreg, f)

In [40]:
best_svm = train_grid_search('SVM', train_svm, X_train_scaled, y_train, X_test, y_test)
with open('../models/svm.pkl', 'wb') as f:
    pickle.dump(best_svm, f)

Fitting 5 folds for each of 5 candidates, totalling 25 fits


D:\PyCharm\hseml-group-project-vassuha\hseml-group-project\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


best CV F1-macro: 0.6298, params: {'C': 0.01}
Model name: SVM
F2-score: 0.64
Accuracy: 0.26
Precision: 0.63
Recall: 0.50


In [41]:
best_random_forest = train_grid_search('Random forest', train_random_forest, X_train_scaled, y_train, X_test, y_test)
with open('../models/random_forest.pkl', 'wb') as f:
    pickle.dump(best_random_forest, f)

Fitting 5 folds for each of 180 candidates, totalling 900 fits
best CV F1-macro: 0.9285, params: {'max_depth': 100, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}
Model name: Random forest
F2-score: 0.94
Accuracy: 0.97
Precision: 0.97
Recall: 0.96


In [42]:
best_xgboost = train_grid_search('XGBoost', train_xgboost, X_train_scaled, y_train, X_test, y_test)
with open('../models/xgboost.pkl', 'wb') as f:
    pickle.dump(best_xgboost, f)

Fitting 5 folds for each of 36 candidates, totalling 180 fits
best CV F1-macro: 0.9360, params: {'learning_rate': 0.1, 'max_depth': 10, 'n_estimators': 1000, 'objective': 'binary:logistic'}
Model name: XGBoost
F2-score: 0.95
Accuracy: 0.97
Precision: 0.97
Recall: 0.96


# Эксперименты с уменьшением размерности

In [43]:
df_reduced = pd.read_csv("../data/processed/f1_strategy_dataset_v4_processed_reduced.csv")

In [44]:
X = df_reduced.drop('PitNextLap', axis=1)
y = df_reduced['PitNextLap']

X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = split(X, y, test_size=0.2, random_state=42)
print(f"Доля пит-стопов в трейне: {y_train.mean():.4f}")
print(f"Доля пит-стопов в тесте: {y_test.mean():.4f}")

Доля пит-стопов в трейне: 0.2539
Доля пит-стопов в тесте: 0.2583


In [45]:
scaler = StandardScaler()
scaler.fit(X_train_reduced)

X_test_reduced_scaled = scaler.transform(X_test_reduced)
X_train_reduced_scaled = scaler.transform(X_train_reduced)

X_train_reduced = X_train_reduced_scaled
X_test_reduced = X_test_reduced_scaled

In [46]:
random_forest_reduced = train_grid_search('Random forest', train_random_forest, X_train_reduced, y_train_reduced, X_test_reduced, y_test_reduced)
with open('../models/random_forest_reduced.pkl', 'wb') as f:
    pickle.dump(best_random_forest, f)

Fitting 5 folds for each of 180 candidates, totalling 900 fits
best CV F1-macro: 0.9038, params: {'max_depth': 100, 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}
Model name: Random forest
F2-score: 0.92
Accuracy: 0.96
Precision: 0.95
Recall: 0.95
